In [ ]:
encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=4)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)
src = torch.rand(13, 31, 512)
out = transformer_encoder.forward(src)
print(out.shape)

In [ ]:
max_seq_len = max([len(n) for n in train_set["content"]])

In [ ]:
embeddings = nn.Embedding(num_embeddings=dict_size+special_tokens, embedding_dim=512, padding_idx=0)
proj_out = nn.Linear(512, dict_size+special_tokens)
positional_encoding = nn.Parameter(torch.zeros(max_seq_len, 512))

In [ ]:
def forward(seq: int):
    seq = seq[:max_seq_len]
    x = embeddings(seq)
    x += positional_encoding[:len(seq), :]
    return proj_out(transformer_encoder(x))

In [ ]:
# Pre training ...

opt = torch.optim.Adam([*transformer_encoder.parameters(), *embeddings.parameters(), *proj_out.parameters(), positional_encoding], lr=0.001)
costs = []

for epoch in range(3):
    n = 0

    for seq, label in zip(train_set["content"], train_set["new_labels"]):
        if type(seq) is not str:
            print(seq)
            continue
        seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
        y_seq = seq.clone()
        #y_seq = F.one_hot(seq, num_classes=501).to(torch.int64)
        seq[random.randint(0, len(seq)-1)] = 0
        out = forward(seq)
        loss = F.cross_entropy(out, y_seq)
        loss.backward()
        opt.step()
        opt.zero_grad()
        costs.append(loss.item())
        print(f"{epoch} : {n} / {len(train_set)} - {float(np.mean(costs)):.2f}", end='\r')
        n+=1
        #if float(np.mean(costs)) < 1.1:
        #    break

In [ ]:
proj_out = nn.Linear(512, 1)
def forward(seq: int):
    seq = seq[:max_seq_len]
    x = embeddings(seq)
    x += positional_encoding[:len(seq), :]
    x = torch.vstack([x, torch.zeros([512])])
    #return proj_out(transformer_encoder(x).mean(axis=0)).sigmoid()
    return proj_out(transformer_encoder(x)[-1,:]).tanh()

opt = torch.optim.Adam([*transformer_encoder.parameters(), *embeddings.parameters(), positional_encoding], lr=0.0001)
opt2 = torch.optim.Adam(proj_out.parameters(), lr=0.001)
costs = []

for epoch in range(2):
    n = 0

    for seq, label in zip(train_set["content"], train_set["new_labels"]):
        if type(seq) is not str:
            print(seq)
            continue
        seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
        #seq[random.randint(0, len(seq))] = 0
    
        if label == "fake":
            y = 1
        else:
            y = -1
        
        out = forward(seq)
        loss = F.mse_loss(out, torch.Tensor([y]))
        costs.append(loss.item())
        loss.backward()
        opt2.step()
        opt2.zero_grad()
        opt.step()
        opt.zero_grad()
        print(f"{epoch} : {n} / {len(train_set.index)} - {float(np.mean(costs)):.2f} - {out.item():.2f} - {label}", end='\r')
        n += 1
        #if n > 130:
        #    break

In [ ]:
true_positive = 0
false_negative = 0
correct = 0

for seq, label in zip(train_set["content"], train_set["new_labels"]):
    seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
    
    y_hat = "fake" if forward(seq).item() > 0.0 else "reliable"
    
    if y_hat == "fake" and label == "fake":
        true_positive += 1
    if y_hat == "fake" and label == "reliable":
        false_negative += 1
    if y_hat == label:
        correct += 1

#print(f"F1: {true_positive / (true_positive + false_negative) * 100}%")
print(f"accuracy: {correct / len(train_set) * 100}%")

In [ ]:
n = 0
for seq, label in zip(train_set["content"], train_set["new_labels"]):
    seq = torch.LongTensor(tokenizer.encode(seq).ids)[:max_seq_len] + special_tokens
    out = forward(seq).item()
    print(f"{out} - {label}")
    if n > 10:
        break
    n+=1